[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/cga_spherical_quadrics/cga_spherical_quadrics.ipynb)

# Spherical Quadrics in a Conformal Flow

This example draws curved regions on a sphere, donuts, figure eights and hourglasses, and carries them all around the sphere by one smooth conformal motion, a flow that keeps angles but stretches and shrinks lengths as it goes. Each region is a quadric of the sphere: the points where a quadratic expression is negative. The flow turns around two fixed points where two circles cross, and after a full turn every region is back where it started.

In the conformal algebra of the sphere, `Cl(3, 1)` with `w` squaring to minus one, a circle on the sphere is a vector, `n + s * mv.w` for the plane `n | x == s` that cuts it out, and a point of the sphere is modelled as a null antivector, the sphere's centre `mv.w.dual()` plus a unit direction. A circle paired with a point, `circle & point`, is positive inside the circle. A quadric is a map from points to planes, `Plane <- Point`, a sum of circle dyads such as `outer * (inner & Point)`, and a point lies inside where `quadric(point) & point < 0`. Two circles meet in the 2-blade of their wedge, whose exponential is a conformal rotor, and a rotor moves a quadric as it moves any map, `versor >> quadric(versor << Point)`.

In the notation of conformal geometry the quadrics read as symmetric $4 \times 4$ matrices on the homogeneous coordinates of the sphere, the conformal motions as the Möbius transformations, the group $SO^+(3, 1)$ acting on them by congruence, and the flow as a one-parameter subgroup, loxodromic in general.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from examples.util.animation import show_animation
from examples.quadrics.cga_spherical_quadrics import render
# The algebra, shared with the example's renderer.
from examples.quadrics.cga_spherical_quadrics.core import ga

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
# A circle on S², the plane that cuts it out.
Plane = ga.gatype.vector()                          # Vector+
# A point of S²: an antivector with intrinsic orientation.
Point = ga.gatype.antivector()                      # Antivector-
Quadric = ga.gatype((Plane, Point))
Bivector = ga.gatype.bivector()                     # Bivector+

## 1. Shapes as sums of circle dyads

A ring is the band between two circles of the same centre, inside where a point is inside one and outside the other: the symmetric dyad of the two circles. A figure eight, Bernoulli's lemniscate, is the dyad of the circle through the pole less a weighted difference of two coordinate dyads. An hourglass is a Cassini oval of two tilted planes tangent at its two foci, pinched at the waist. A quadric moves by a versor as any map does, `versor >> quadric(versor << Point)`: the point pulled back, the plane pushed forward.

In [ ]:
# The conical donut's tilt and width, and its turn into view; the lemniscate's spread; the hourglass's tilt and
# waist; and the three shapes' colours.
donut_tilt, donut_width, donut_view = 0.55, 0.22, -0.20
lemniscate_spread = 0.38
hourglass_tilt, hourglass_waist = np.radians(24.0), 0.010
colors = ["#38bdf8", "#a855f7", "#f59e0b"]

In [ ]:
def make_spherical_donut(r_core: float, r_tube: float) -> Quadric:
    """A spherical donut is the band between the circles at the outer and inner edges of a ring on S², inside where
    a point lies inside one circle and outside the other."""
    outer = mv.z + np.cos(r_core + r_tube) * mv.w                              # [] Plane+
    inner = mv.z + np.cos(r_core - r_tube) * mv.w                              # [] Plane+
    return 0.5 * (outer * (inner & Point) + inner * (outer & Point))


def make_conical_donut(tilt: float, width: float) -> Quadric:
    """A conical donut, a limaçon on S², is a ring whose central hole closes in a sharp conical apex at the pole."""
    # A circle through the pole, tilted toward x.
    through_pole = mv.w + mv.z + tilt * mv.x                                   # [] Plane+
    return through_pole * (through_pole & Point) - width**2 * (mv.x * (mv.x & Point) + mv.y * (mv.y & Point))


def make_bernoulli_lemniscate(spread: float) -> Quadric:
    """The lemniscate of Bernoulli on S² is a figure eight: the dyad of the point circle at the pole less a weighted
    difference of the dyads of the planes x and y."""
    pole = mv.w + mv.z                                                         # [] Plane+
    return pole * (pole & Point) - 2.0 * spread**2 * (mv.x * (mv.x & Point) - mv.y * (mv.y & Point))


def make_spherical_hourglass(tilt: float, waist: float) -> Quadric:
    """An hourglass on S² is a Cassini oval of two foci tilted equally either way along y: two symmetric bulbs
    joined by a narrow waist."""
    # The plane z turned toward y and away from it by the same angle; each, with w, is the plane tangent to S² at a
    # focus.
    first_tangent = mv.w + ((mv.zy * (-tilt / 2)).exp() >> mv.z)               # [] Plane+
    second_tangent = mv.w + ((mv.zy * (tilt / 2)).exp() >> mv.z)               # [] Plane+
    return (0.5 * (first_tangent * (second_tangent & Point) + second_tangent * (first_tangent & Point))
            - waist * mv.w * (mv.w & Point))


# The donut tilted into view.
view_tilt = (mv.xz * donut_view).exp()                                         # [] Rotor+
conical_donut = view_tilt >> make_conical_donut(donut_tilt, donut_width)(view_tilt << Point)   # [] Plane+ <- Point-
lemniscate = make_bernoulli_lemniscate(lemniscate_spread)                      # [] Plane+ <- Point-
hourglass = make_spherical_hourglass(hourglass_tilt, hourglass_waist)          # [] Plane+ <- Point-
quadrics = stack([conical_donut, lemniscate, hourglass])                       # [shapes] Plane+ <- Point-

In [ ]:
print("the lemniscate, a map from points to planes:\n", lemniscate.coefficient_table())

## 2. The flow

Two circles off the pole, each a tilted plane with its radius, meet in a 2-blade; its exponential at growing phases is a one-parameter family of conformal rotors, and each carries every shape at once. After a full turn of the phase the shapes are back where they started. The animation shows the front hemisphere, each shape in its colour where a point is inside it.

In [ ]:
# The two circles: each the plane z tilted toward x or y, and its radius; the frames of one turn, the
# pixels across, and a frame's duration in milliseconds.
first_tilt, first_radius = 0.35, np.radians(48.0)
second_tilt, second_radius = 0.40, np.radians(52.0)
n_frames, resolution, frame_duration = 60, 300, 33

In [ ]:
def flow(quadrics: Quadric, generator: Bivector, phases: np.ndarray) -> Quadric:
    """The quadrics carried by (generator * (phase / 2)).exp(), batched as (phases, quadrics)."""
    rotors = (generator * (phases / 2.0)).exp()[:, None]
    return rotors >> quadrics(rotors << Point)


# Each circle is the vector of its plane with the cosine of its radius on w.
first = ((mv.zx * (-first_tilt / 2)).exp() >> mv.z) + mv.w * np.cos(first_radius)      # [] Plane+
second = ((mv.zy * (-second_tilt / 2)).exp() >> mv.z) + mv.w * np.cos(second_radius)   # [] Plane+
generator = (first ^ second).normalized()                                              # [] Bivector+
world = flow(quadrics, generator, np.linspace(0.0, 2.0 * np.pi, n_frames, endpoint=False))   # [frames, shapes] Plane+ <- Point-
show_animation(render.vortex_frames(world, colors, resolution), "cga_trio_vortex", frame_duration)

In [ ]:
# checks
# The flow is conformal, so points of the sphere stay null; and a full turn brings every quadric back.
def point(xyz: np.ndarray) -> Point:
    """The points of S² at `[..., 3]` unit coordinates: the sphere's centre plus their unit directions."""
    return mv.w.dual() + (stack((mv.x.dual(), mv.y.dual(), mv.z.dual())) * xyz).sum(axis=-1)


probes = point(np.array([[0.6, 0.0, 0.8], [0.0, -0.28, 0.96], [0.36, 0.48, 0.8]]))
carried = (generator * 0.4).exp() >> probes
np.testing.assert_allclose((carried | carried).to_array(), 0.0, atol=1e-12)
returned = flow(quadrics, generator, np.array([2.0 * np.pi]))[0]
np.testing.assert_allclose((returned(probes[:, None]) & probes[:, None]).to_array(),
                           (quadrics(probes[:, None]) & probes[:, None]).to_array(), atol=1e-6)